In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [ ]:
# ---> LOAD PROJECT UTILITIES & INIT NOTEBOOK WIDGETS <---

%run /Workspace/consolidated_pipeline/1_setup/utilities


In [ ]:
dbutils.widget.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sports-consolidated-dp/{data_source}/*.csv'

In [ ]:
# BRONZE

df = (
    spark.read.format('csv')
        .option('header', True)
        .option('inferSchema', True)
        .load(base_path)
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', _'metadata.file_name', '_metadata.file_size')
)

In [ ]:
df.write\
    .format('delta') \
    .option('delta.enableDataChange', 'true')\
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [ ]:
# SILVER
df_bronze = spark.sql(f'SELECT * from {catalog}.{bronze_schema}.{data_source};')

In [ ]:
# ---> NORMALIZE MONTH FIELD <---

df_bronze.select('month').distinct().show()

# +----------+
# |     month|
# +----------+
# |2025/07/01|
# |01/08/2025|
# |2025/09/01|
# |2025-10-01|
# |2025-11-01|
# |2025-12-01|
# |2025-07-01|
# |2025-08-01|
# |2025-09-01|
# |2025/11/01|
# |2025/08/01|
# |01-09-2025|
# |2025/10/01|
# |01/12/2025|
# |01/09/2025|
# |01-12-2025|
# |01-08-2025|
# |01/10/2025|
# +----------+